# Bronze to Silver: Geospatial Risk Intelligence Pipeline


> **Recommended runtime: Medium** for the City of San Diego AOI (~357K buildings, about 13 minutes; Large saves only a minute). Use **Large** for the county AOI (~1.03M buildings, about 35 minutes) — raster zonal stats + spatial KNN + cached-buildings shuffle.
This notebook transforms Bronze-layer Iceberg tables into Silver-layer tables containing
spatially joined, analytically useful datasets using Wherobots Cloud and Apache Sedona.

## Silver tables produced

| Silver Table | Operation | Inputs |
|---|---|---|
| `org_catalog.silver.asset_wildfire_exposure` | Zonal Statistics (raster → vector) | USFS Wildfire Burn Prob / Flame Length + Overture Buildings |
| `org_catalog.silver.asset_flood_exposure` | Zonal Statistics + Temporal Aggregation | OPERA DSWx-S1 (SAR flood) + Overture Buildings |
| `org_catalog.silver.asset_weather_density` | KNN Join + Buffer Aggregation | NOAA SWDI (hail, structure, tvs) + Overture Buildings |
| `org_catalog.silver.asset_enriched` | Conflation Join | All three above |

## Bronze inputs

| Dataset | Catalog Table | Type |
|---|---|---|
| Buildings | `wherobots_open_data.overture_maps_foundation.buildings_building` | Vector |
| Wildfire Burn Probability | `org_catalog.wildfire_risk.burn_probability_conus` | Raster |
| Wildfire Flame Length | `org_catalog.wildfire_risk.conditional_flame_length_conus` | Raster |
| OPERA DSWx-S1 Flood | `org_catalog.opera.dswx_s1` | Raster |
| NOAA SWDI Hail | `org_catalog.noaa_swdi.hail` | Vector |
| NOAA SWDI Structure | `org_catalog.noaa_swdi.structure` | Vector |
| NOAA SWDI TVS | `org_catalog.noaa_swdi.tvs` | Vector |

## Notebook sequence

```
raw-to-bronze.ipynb  →  bronze-to-silver.ipynb  →  silver-to-gold.ipynb
                             (you are here)
```

## Prerequisites
- Wherobots Cloud runtime with Apache Sedona
- Bronze tables populated (run `raw-to-bronze.ipynb` first)
- `wkls` Python library (`pip install wkls`)

## 0. Configuration & Session Setup

In [1]:
from sedona.spark import *
from pyspark.sql import functions as F
from datetime import datetime, date
import wkls

# ── Pipeline Parameters ──────────────────────────────────────────────────────
# Geographic scope: the City of San Diego, California via wkls (~357K buildings).
# This is the workshop's reference AOI: about 13 minutes on a Medium runtime, which
# fits a live session. For the whole county (~1.03M buildings, the eastern
# wildland-urban interface around Poway, Ramona and Julian) use
# wkls.us.ca.sandiegocounty.wkt() on a Large runtime, about 35 minutes.
AOI_WKT = wkls.us.ca.sandiego.wkt()
print(f"AOI loaded: City of San Diego, California ({len(AOI_WKT):,} chars)")

# Temporal windows — per source, since datasets have different coverage periods
# OPERA DSWx-S1 flood (Sentinel-1 SAR, Dec 2025 – Mar 2026 storm season)
FLOOD_WINDOW_START = "2025-12-01"
FLOOD_WINDOW_END   = "2026-03-31"

# NOAA SWDI severe weather (hail, structure, TVS)
WEATHER_WINDOW_START = "2025-01-01"
WEATHER_WINDOW_END   = "2025-12-31"  # the radar tables end here (bootstrap loads the 2024 and 2025 annual files); check datetime.end on the table before extending

# Wildfire risk classification thresholds (burn probability)
WF_THRESHOLD_EXTREME   = 0.05
WF_THRESHOLD_VERY_HIGH = 0.01
WF_THRESHOLD_HIGH      = 0.002
WF_THRESHOLD_MODERATE  = 0.0005

# ── Table References ─────────────────────────────────────────────────────────
# Bronze / Source tables
BUILDINGS_TABLE   = "wherobots_open_data.overture_maps_foundation.buildings_building"
WILDFIRE_BP_TABLE = "org_catalog.wildfire_risk.burn_probability_conus"
WILDFIRE_FL_TABLE = "org_catalog.wildfire_risk.conditional_flame_length_conus"
FLOOD_TABLE       = "org_catalog.opera.dswx_s1"
SWDI_HAIL_TABLE   = "org_catalog.noaa_swdi.hail"
SWDI_STRUCT_TABLE = "org_catalog.noaa_swdi.structure"
SWDI_TVS_TABLE    = "org_catalog.noaa_swdi.tvs"

# Silver output database (in org_catalog)
SILVER_DB = "silver"

print(f"Flood window:       {FLOOD_WINDOW_START} → {FLOOD_WINDOW_END}")
print(f"Weather window:     {WEATHER_WINDOW_START} → {WEATHER_WINDOW_END}")

AOI loaded: City of San Diego, California (137,478 chars)
Flood window:       2025-12-01 → 2026-03-31
Weather window:     2025-01-01 → 2026-03-25


In [2]:
# ── Initialize Wherobots Sedona Session ───────────────────────────────────────
config = SedonaContext.builder().getOrCreate()
config.sparkContext.setLogLevel("ERROR")  # hide Spark WARN chatter (function registration, plan truncation, window partitioning); real errors still surface
sedona = SedonaContext.create(config)

# Create the Silver database if it doesn't exist
sedona.sql(f"CREATE DATABASE IF NOT EXISTS org_catalog.{SILVER_DB}")

print("Sedona session initialized ✓")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
Setting Spark log level to "WARN".
26/09/29 02:42:21 INFO core/src/lib.rs: Sedona native acceleration engine v0.15.3 ready


Sedona session initialized ✓


## 1. Explore Bronze Tables

Before transforming, verify the Bronze layer is populated and inspect row counts and schema.

In [3]:
# ── Inspect source tables ─────────────────────────────────────────────────────
source_tables = [
    BUILDINGS_TABLE,
    WILDFIRE_BP_TABLE,
    WILDFIRE_FL_TABLE,
    FLOOD_TABLE,
    SWDI_HAIL_TABLE,
    SWDI_STRUCT_TABLE,
    SWDI_TVS_TABLE,
]

for table in source_tables:
    try:
        df = sedona.table(table)
        count = df.count()
        print(f"  ✓ {table:70s} rows: {count:>12,}")
    except Exception as e:
        print(f"  ✗ {table:70s} ERROR: {e}")

  ✓ wherobots_open_data.overture_maps_foundation.buildings_building        rows: 2,533,842,612
  ✓ org_catalog.wildfire_risk.burn_probability_conus                       rows:      970,268
  ✓ org_catalog.wildfire_risk.conditional_flame_length_conus               rows:      970,268
  ✓ org_catalog.opera.dswx_s1                                              rows:    2,395,168
  ✓ org_catalog.noaa_swdi.hail                                             rows:   25,784,125
  ✓ org_catalog.noaa_swdi.structure                                        rows:   83,258,366
  ✓ org_catalog.noaa_swdi.tvs                                              rows:       92,704


In [4]:
# ── Load source tables scoped to AOI ──────────────────────────────────────────
# Buildings: filter by AOI polygon
buildings = sedona.sql(f"""
    SELECT id, geometry, height, num_floors, class, subtype, names.primary AS building_name
    FROM {BUILDINGS_TABLE}
    WHERE ST_Intersects(geometry, ST_GeomFromText('{AOI_WKT}'))
""")
buildings.createOrReplaceTempView("buildings")

# Wildfire rasters: filter tiles overlapping AOI
wildfire_bp = sedona.table(WILDFIRE_BP_TABLE).filter(f"RS_Intersects(raster, ST_GeomFromText('{AOI_WKT}'))")
wildfire_bp.createOrReplaceTempView("wildfire_bp")

wildfire_fl = sedona.table(WILDFIRE_FL_TABLE).filter(f"RS_Intersects(raster, ST_GeomFromText('{AOI_WKT}'))")
wildfire_fl.createOrReplaceTempView("wildfire_fl")

# OPERA DSWx-S1 Flood: load B01_WTR (water classification) band only, filtered to AOI.
# B01_WTR values: 0=not water, 1=open water, 3=inundated vegetation, 250=HAND (terrain)
# mask, 251=layover/shadow mask, 255=no data. The two masks mean "not observed", not
# water, so they are set to no-data here; RS_ZonalStats (excludeNoData=true) then ignores
# them instead of reporting 250 or 251 as the maximum water class of a footprint.
flood_wtr = (
    sedona.table(FLOOD_TABLE)
    .filter(f"band = 'B01_WTR' AND RS_Intersects(raster, ST_GeomFromText('{AOI_WKT}'))")
    .withColumn("raster", F.expr("RS_MapAlgebra(raster, 'B', 'out = (rast[0] == 250 || rast[0] == 251) ? 255 : rast[0];', 255)"))
)
flood_wtr.createOrReplaceTempView("flood_b01_wtr")

# SWDI tables are loaded per-event-type in section 4 (KNN join)

print(f"Buildings:      {buildings.count():>12,} rows (AOI)")
print(f"Wildfire BP:    {wildfire_bp.count():>12,} tiles")
print(f"Wildfire FL:    {wildfire_fl.count():>12,} tiles")
print(f"Flood B01_WTR:  {flood_wtr.count():>12,} tiles")

Buildings:           357,263 rows (AOI)


Wildfire BP:             115 tiles


Wildfire FL:             115 tiles


Flood B01_WTR:         8,852 tiles


## 2. Silver Table: `asset_wildfire_exposure`

**Operation**: Zonal Statistics (raster → vector)

For each building footprint, compute the mean and max burn probability from USFS wildfire risk rasters,
plus the mean conditional flame length. Classify each asset into a wildfire risk tier.

> **Sedona Functions**: `RS_ZonalStats(raster, geometry, statType)`, `RS_Intersects`
> 
> **Note**: RS_ZonalStats automatically handles CRS transformation between raster and geometry

In [5]:
# ── 2a. Compute wildfire exposure via zonal stats ────────────────────────────
# Join building footprints with wildfire raster tiles using RS_Intersects,
# then extract burn probability and flame length statistics per asset.
# RS_ZonalStats(raster, geometry, statType) handles CRS alignment automatically.

wildfire_exposure = sedona.sql(f"""
    WITH bp AS (
        SELECT
            b.id AS asset_id,
            b.geometry,
            b.height,
            b.num_floors,
            b.class,
            avg(RS_ZonalStats(w.raster, b.geometry, 1, 'mean', true)) AS burn_prob_mean,
            max(RS_ZonalStats(w.raster, b.geometry, 1, 'max', true))  AS burn_prob_max
        FROM buildings b
        JOIN wildfire_bp w
            ON RS_Intersects(w.raster, b.geometry)
        GROUP BY b.id, b.geometry, b.height, b.num_floors, b.class
    ),
    fl AS (
        SELECT
            b.id AS asset_id,
            avg(RS_ZonalStats(w.raster, b.geometry, 1, 'mean', true)) AS flame_length_mean
        FROM buildings b
        JOIN wildfire_fl w
            ON RS_Intersects(w.raster, b.geometry)
        GROUP BY b.id
    )
    SELECT
        bp.asset_id,
        'building' AS asset_type,
        bp.geometry,
        bp.height,
        bp.num_floors,
        bp.class,
        bp.burn_prob_mean,
        bp.burn_prob_max,
        COALESCE(fl.flame_length_mean, 0.0) AS flame_length_mean,
        CASE
            WHEN bp.burn_prob_mean >= {WF_THRESHOLD_EXTREME}   THEN 'extreme'
            WHEN bp.burn_prob_mean >= {WF_THRESHOLD_VERY_HIGH} THEN 'very_high'
            WHEN bp.burn_prob_mean >= {WF_THRESHOLD_HIGH}      THEN 'high'
            WHEN bp.burn_prob_mean >= {WF_THRESHOLD_MODERATE}  THEN 'moderate'
            ELSE 'low'
        END AS wildfire_risk_class,
        current_timestamp() AS computed_at
    FROM bp
    LEFT JOIN fl ON bp.asset_id = fl.asset_id
""")

WILDFIRE_SILVER = f"org_catalog.{SILVER_DB}.asset_wildfire_exposure"
wildfire_exposure.writeTo(WILDFIRE_SILVER).createOrReplace()
wildfire_exposure = sedona.table(WILDFIRE_SILVER)

print(f"Wildfire exposure rows: {wildfire_exposure.count():,}")
wildfire_exposure.groupBy("wildfire_risk_class").count().orderBy("count", ascending=False).show()

Wildfire exposure rows: 357,263


,wildfire_risk_class,count
0,low,328416
1,moderate,20029
2,high,8583
3,very_high,235


## 3. Silver Table: `asset_flood_exposure`

**Operation**: Zonal Statistics, one week at a time → append to Iceberg (raster → vector)

For each building, compute the max DSWx-S1 **B01_WTR** classification per ISO week using
`allTouched=true` so buildings smaller than 30 m still capture intersecting flood pixels.

The loop processes **one week per iteration**, writing results to Iceberg via
`createOrReplace` (first week) then `append` (subsequent weeks). This keeps shuffle small
(only one week of raster tiles at a time) and gives free checkpointing — if a week fails,
restart from there.

The output is one row per **(asset, week)** — temporal rollup is deferred to the Gold layer.

| Column | Description |
|---|---|
| `flood_week` | Start of the ISO week |
| `flood_max_wtr_class` | Max WTR class that week (0=not water, 1=open water, 3=inundated vegetation) |

> **Sedona Functions**: `RS_ZonalStats(raster, geometry, band, statType, excludeNoData)`,
> `RS_Intersects`

In [6]:
sedona.sql('SELECT RS_Metadata(raster) as metadata FROM flood_b01_wtr').show(5, 0)

,metadata
0,"Row(upperLeftX=492120.0, upperLeftY=3630900.0, gridWidth=128, gridHeight=128, scaleX=30.0, scaleY=-30.0, skewX=0.0, skewY=0.0, srid=32611, numSampleDimensions=1, tileWidth=128, tileHeight=128)"
1,"Row(upperLeftX=503640.0, upperLeftY=3661620.0, gridWidth=128, gridHeight=128, scaleX=30.0, scaleY=-30.0, skewX=0.0, skewY=0.0, srid=32611, numSampleDimensions=1, tileWidth=128, tileHeight=128)"
2,"Row(upperLeftX=472920.0, upperLeftY=3615540.0, gridWidth=128, gridHeight=128, scaleX=30.0, scaleY=-30.0, skewX=0.0, skewY=0.0, srid=32611, numSampleDimensions=1, tileWidth=128, tileHeight=128)"
3,"Row(upperLeftX=484440.0, upperLeftY=3653940.0, gridWidth=128, gridHeight=128, scaleX=30.0, scaleY=-30.0, skewX=0.0, skewY=0.0, srid=32611, numSampleDimensions=1, tileWidth=128, tileHeight=128)"
4,"Row(upperLeftX=503820.0, upperLeftY=3615540.0, gridWidth=128, gridHeight=128, scaleX=30.0, scaleY=-30.0, skewX=0.0, skewY=0.0, srid=32611, numSampleDimensions=1, tileWidth=128, tileHeight=128)"


In [7]:
# ── 3a. Compute weekly flood exposure, one week at a time → Iceberg ───────────
# Process each ISO week independently to keep shuffle small (buildings × 1 week
# of raster tiles). Results are appended to Iceberg week by week — free
# checkpointing if a week fails.
# allTouched=true (5th arg) ensures partially-overlapping pixels are included.
# B01_WTR water classification: 0=not water, 1=open water, 3=inundated vegetation (masks removed in cell 6)

from datetime import date, timedelta

FLOOD_SILVER = f"org_catalog.{SILVER_DB}.asset_flood_exposure"

# Generate ISO week start dates covering the flood window
start = date.fromisoformat(FLOOD_WINDOW_START)
end   = date.fromisoformat(FLOOD_WINDOW_END)

# Align to Monday (ISO week start)
week_start = start - timedelta(days=start.weekday())
weeks = []
while week_start <= end:
    weeks.append(week_start)
    week_start += timedelta(days=7)

print(f"Flood window: {FLOOD_WINDOW_START} → {FLOOD_WINDOW_END}")
print(f"Processing {len(weeks)} weeks: {weeks[0]} → {weeks[-1]}\n")

first = True
for i, wk in enumerate(weeks):
    wk_end = wk + timedelta(days=6)
    # Clamp to the actual flood window boundaries
    wk_start_clamped = max(wk, start)
    wk_end_clamped   = min(wk_end, end)

    df = sedona.sql(f"""
        SELECT
            b.id                                                        AS asset_id,
            'building'                                                  AS asset_type,
            b.geometry,
            DATE('{wk.isoformat()}')                                    AS flood_week,
            MAX(RS_ZonalStats(w.raster, b.geometry, 1, 'max', true))    AS flood_max_wtr_class,
            DATE('{FLOOD_WINDOW_START}')                                AS observation_window_start,
            DATE('{FLOOD_WINDOW_END}')                                  AS observation_window_end,
            current_timestamp()                                         AS computed_at
        FROM buildings b
        JOIN flood_b01_wtr w
            ON RS_Intersects(w.raster, b.geometry)
        WHERE w.acq_date BETWEEN DATE('{wk_start_clamped.isoformat()}')
                              AND DATE('{wk_end_clamped.isoformat()}')
        GROUP BY b.id, b.geometry
    """)

    if first:
        df.writeTo(FLOOD_SILVER).createOrReplace()
        first = False
    else:
        df.writeTo(FLOOD_SILVER).append()

    print(f"  ✓ week {i+1}/{len(weeks)}  {wk_start_clamped} → {wk_end_clamped}")

total = sedona.table(FLOOD_SILVER).count()
print(f"\n✓ Wrote {total:,} rows to {FLOOD_SILVER}  (asset × week)")

Flood window: 2025-12-01 → 2026-03-31
Processing 18 weeks: 2025-12-01 → 2026-03-30



  ✓ week 1/18  2025-12-01 → 2025-12-07


  ✓ week 2/18  2025-12-08 → 2025-12-14


  ✓ week 3/18  2025-12-15 → 2025-12-21


  ✓ week 4/18  2025-12-22 → 2025-12-28


...

  ✓ week 13/18  2026-02-23 → 2026-03-01


  ✓ week 14/18  2026-03-02 → 2026-03-08


  ✓ week 15/18  2026-03-09 → 2026-03-15


  ✓ week 16/18  2026-03-16 → 2026-03-22


  ✓ week 17/18  2026-03-23 → 2026-03-29
  ✓ week 18/18  2026-03-30 → 2026-03-31

✓ Wrote 6,073,471 rows to org_catalog.silver.asset_flood_exposure  (asset × week)


In [8]:
# ── 3b. Verify silver.asset_flood_exposure ────────────────────────────────────
FLOOD_SILVER = f"org_catalog.{SILVER_DB}.asset_flood_exposure"

flood_check = sedona.table(FLOOD_SILVER)
print(f"Rows:           {flood_check.count():,}")
print(f"Distinct assets:{flood_check.select('asset_id').distinct().count():,}")
print(f"Distinct weeks: {flood_check.select('flood_week').distinct().count():,}")
flood_check.groupBy("flood_week").count().orderBy("flood_week").show(50, truncate=False)

Rows:           6,073,471


Distinct assets:357,263


Distinct weeks: 17


,flood_week,count
0,2025-12-01,357263
1,2025-12-08,357263
2,2025-12-15,357263
3,2025-12-22,357263
4,2025-12-29,357263
5,2026-01-05,357263
6,2026-01-12,357263
7,2026-01-19,357263
8,2026-01-26,357263
9,2026-02-02,357263


## 4. Silver Table: `asset_weather_density`

**Operation**: KNN Join with search radius + Spheroidal Distance (vector → vector)

For each building, find the K nearest NOAA SWDI severe weather events **per event type**
(hail, structure, TVS) using WherobotsDB's KNN join with `search_radius` to cap the search
at 25 km. Uses `use_sphere = true` so both the KNN search and the radius are in meters.

> **WherobotsDB**: `ST_KNN(R, S, k, use_sphere, search_radius)` — 5th param limits search distance
>
> **Sources**: `org_catalog.noaa_swdi.hail`, `.structure`, `.tvs` — queried separately per event type

In [9]:
# ── 4a. KNN join per SWDI event type → append to Iceberg ─────────────────────
# Use ST_KNN with search_radius (Wherobots extension) to limit neighbor search
# to 25 km. use_sphere=TRUE means both the KNN distance and search_radius are
# in meters. Pre-filter each SWDI table to AOI + temporal window first, and
# MATERIALIZE the filtered subset to an Iceberg staging table before the join:
# Sedona warns that a filter pushed down on the object side of a KNN join
# ("JoinQueryDetector: Filter pushdown detected on the object side of a KNN
# join") can return incorrect neighbours. Joining against a materialized table
# leaves nothing to push down.

K_NEAREST      = 10
SEARCH_RADIUS  = 25000    # 25 km in meters (search_radius param)
BUFFER_NEAR_M  = 5000     # 5 km threshold for aggregation
BUFFER_FAR_M   = 25000    # 25 km threshold for aggregation

KNN_STAGING = f"org_catalog.{SILVER_DB}.weather_knn_staging"

swdi_sources = {
    "hail":      (SWDI_HAIL_TABLE,   "SEVPROB",     "MAXSIZE"),
    "structure": (SWDI_STRUCT_TABLE,  "MAX_REFLECT", "VIL"),
    "tvs":       (SWDI_TVS_TABLE,    "MXDV",        "MAX_SHEAR"),
}

first = True
for event_type, (table, severity_col, magnitude_col) in swdi_sources.items():
    # Step 1: pre-filter to AOI + temporal window and materialize to Iceberg
    # (replaced on every run), then join against the materialized table.
    aoi_table = f"org_catalog.{SILVER_DB}.swdi_{event_type}_aoi"
    sedona.sql(f"""
        SELECT geometry, ZTIME, {severity_col}, {magnitude_col}
        FROM {table}
        WHERE ZTIME BETWEEN TIMESTAMP('{WEATHER_WINDOW_START}') AND TIMESTAMP('{WEATHER_WINDOW_END}')
          AND ST_Intersects(geometry, ST_GeomFromText('{AOI_WKT}'))
    """).writeTo(aoi_table).createOrReplace()
    view_name = f"swdi_{event_type}_aoi"
    sedona.table(aoi_table).createOrReplaceTempView(view_name)
    cnt = sedona.sql(f"SELECT COUNT(*) AS c FROM {view_name}").collect()[0]["c"]
    print(f"  {event_type}: {cnt:,} events in AOI (materialized to {aoi_table})")

    # Step 2: KNN join with use_sphere=TRUE and search_radius=25km
    df = sedona.sql(f"""
        SELECT
            b.id                                              AS asset_id,
            b.geometry                                        AS asset_geometry,
            '{event_type}'                                    AS event_type,
            e.ZTIME                                           AS event_time,
            e.{severity_col}                                  AS severity,
            e.{magnitude_col}                                 AS magnitude,
            ST_DistanceSpheroid(b.geometry, e.geometry)        AS distance_m
        FROM buildings b
        JOIN {view_name} e
            ON ST_KNN(b.geometry, e.geometry, {K_NEAREST}, true, {SEARCH_RADIUS})
    """)

    # Step 3: write to Iceberg — createOrReplace first, append after
    if first:
        df.writeTo(KNN_STAGING).createOrReplace()
        first = False
    else:
        df.writeTo(KNN_STAGING).append()
    print(f"  ✓ {event_type} written to {KNN_STAGING}")

total = sedona.table(KNN_STAGING).count()
print(f"\nTotal staging rows: {total:,}")
sedona.table(KNN_STAGING).show(10, truncate=False)

  hail: 11 events in AOI (materialized to org_catalog.silver.swdi_hail_aoi)


  ✓ hail written to org_catalog.silver.weather_knn_staging


  structure: 606 events in AOI (materialized to org_catalog.silver.swdi_structure_aoi)


  ✓ structure written to org_catalog.silver.weather_knn_staging
  tvs: 0 events in AOI (materialized to org_catalog.silver.swdi_tvs_aoi)


  ✓ tvs written to org_catalog.silver.weather_knn_staging

Total staging rows: 6,062,544


,asset_id,asset_geometry,event_type,event_time,severity,magnitude,distance_m
0,1ba6245e-3e82-443c-86d1-93db3dc636db,,hail,2025-05-31 19:38:25,30.0,0.75,6595.170659546761
1,1ba6245e-3e82-443c-86d1-93db3dc636db,,hail,2025-05-31 19:31:54,0.0,0.5,11863.11933635097
2,1ba6245e-3e82-443c-86d1-93db3dc636db,,hail,2025-05-31 19:58:24,-999.0,-999.0,16413.24635960468
3,1ba6245e-3e82-443c-86d1-93db3dc636db,,hail,2025-05-31 20:03:40,-999.0,-999.0,17596.015080353627
4,1ba6245e-3e82-443c-86d1-93db3dc636db,,hail,2025-05-31 20:08:50,-999.0,-999.0,20264.799549540545
5,caad47c2-6ed1-49b4-bada-eb27333f5378,,hail,2025-05-31 19:38:25,30.0,0.75,6608.598877751491
6,caad47c2-6ed1-49b4-bada-eb27333f5378,,hail,2025-05-31 19:31:54,0.0,0.5,11884.203809510816
7,caad47c2-6ed1-49b4-bada-eb27333f5378,,hail,2025-05-31 19:58:24,-999.0,-999.0,16404.126464758472
8,caad47c2-6ed1-49b4-bada-eb27333f5378,,hail,2025-05-31 20:03:40,-999.0,-999.0,17585.268216508426
9,caad47c2-6ed1-49b4-bada-eb27333f5378,,hail,2025-05-31 20:08:50,-999.0,-999.0,20251.57944950087


In [10]:
# ── 4b. Aggregate KNN staging → asset_weather_density ────────────────────────
# Read from the Iceberg staging table (no in-memory cache needed).

WEATHER_SILVER = f"org_catalog.{SILVER_DB}.asset_weather_density"

weather_density = sedona.sql(f"""
    SELECT
        asset_id,
        'building'                                                 AS asset_type,
        asset_geometry                                             AS geometry,

        SUM(CASE WHEN distance_m <= {BUFFER_NEAR_M} THEN 1 ELSE 0 END)   AS event_count_5km,
        SUM(CASE WHEN distance_m <= {BUFFER_FAR_M} THEN 1 ELSE 0 END)    AS event_count_25km,

        MIN(distance_m)                                            AS nearest_event_dist_m,

        MIN(CASE WHEN event_type = 'hail'      THEN distance_m END) AS nearest_hail_m,
        MIN(CASE WHEN event_type = 'structure'  THEN distance_m END) AS nearest_structure_m,
        MIN(CASE WHEN event_type = 'tvs'        THEN distance_m END) AS nearest_tvs_m,

        SUM(CASE WHEN event_type = 'hail'      AND distance_m <= {BUFFER_FAR_M} THEN 1 ELSE 0 END) AS hail_count_25km,
        SUM(CASE WHEN event_type = 'structure'  AND distance_m <= {BUFFER_FAR_M} THEN 1 ELSE 0 END) AS structure_count_25km,
        SUM(CASE WHEN event_type = 'tvs'        AND distance_m <= {BUFFER_FAR_M} THEN 1 ELSE 0 END) AS tvs_count_25km,

        -- Per-event-type max severity (units differ: SEVPROB 0-100, MAX_REFLECT dBZ, MXDV knots)
        MAX(CASE WHEN event_type = 'hail'      THEN severity END)  AS max_hail_sevprob,
        MAX(CASE WHEN event_type = 'structure'  THEN severity END) AS max_structure_reflectivity,
        MAX(CASE WHEN event_type = 'tvs'        THEN severity END) AS max_tvs_delta_v,

        DATE('{WEATHER_WINDOW_START}')                             AS observation_window_start,
        DATE('{WEATHER_WINDOW_END}')                               AS observation_window_end,
        current_timestamp()                                        AS computed_at

    FROM {KNN_STAGING}
    GROUP BY asset_id, asset_geometry
""")

weather_density.writeTo(WEATHER_SILVER).createOrReplace()
row_count = sedona.table(WEATHER_SILVER).count()
print(f"✓ Wrote {row_count:,} rows to {WEATHER_SILVER}")

✓ Wrote 357,263 rows to org_catalog.silver.asset_weather_density


## 5. Silver Table: `asset_enriched`

**Operation**: Conflation Join — Merge all three hazard exposure layers onto the base asset table

This is the unified Silver table that Gold will consume. Each row is an asset with all hazard signals attached. Uses `LEFT JOIN` so assets with no exposure in a given hazard still appear (with nulls).

In [11]:
# ── 5a. Conflate wildfire + weather exposure onto base assets ─────────────────
# LEFT JOIN ensures every building in scope appears even if it has no
# wildfire/weather exposure — those columns will be null.
#
# NOTE: Flood exposure (asset_flood_exposure) is NOT joined here because it
# contains weekly rows (one per asset × week). The Gold layer joins and
# aggregates flood data directly to preserve temporal granularity for scoring.

WILDFIRE_SILVER = f"org_catalog.{SILVER_DB}.asset_wildfire_exposure"
WEATHER_SILVER  = f"org_catalog.{SILVER_DB}.asset_weather_density"

asset_enriched = sedona.sql(f"""
    SELECT
        b.id                               AS asset_id,
        'building'                         AS asset_type,
        b.geometry,
        b.class                            AS building_class,
        b.height,
        b.num_floors,

        -- Wildfire exposure
        w.burn_prob_mean,
        w.burn_prob_max,
        w.flame_length_mean,
        w.wildfire_risk_class,

        -- Severe weather density (KNN-derived)
        s.event_count_5km,
        s.event_count_25km,
        s.nearest_event_dist_m,
        s.nearest_hail_m,
        s.nearest_structure_m,
        s.nearest_tvs_m,
        s.hail_count_25km,
        s.structure_count_25km,
        s.tvs_count_25km,
        s.max_hail_sevprob,
        s.max_structure_reflectivity,
        s.max_tvs_delta_v,

        -- Data coverage flags (null = no data, not zero risk)
        CASE WHEN w.asset_id IS NOT NULL THEN TRUE ELSE FALSE END AS has_wildfire_data,
        CASE WHEN s.asset_id IS NOT NULL THEN TRUE ELSE FALSE END AS has_weather_data,

        -- Temporal window
        DATE('{WEATHER_WINDOW_START}')     AS weather_window_start,
        DATE('{WEATHER_WINDOW_END}')       AS weather_window_end,

        current_timestamp()                AS computed_at

    FROM buildings b
    LEFT JOIN {WILDFIRE_SILVER} w
        ON b.id = w.asset_id
    LEFT JOIN {WEATHER_SILVER} s
        ON b.id = s.asset_id
""")

asset_enriched.cache()
print(f"Asset enriched rows: {asset_enriched.count():,}")
asset_enriched.printSchema()

Asset enriched rows: 357,263
root
 |-- asset_id: string (nullable = true)
 |-- asset_type: string (nullable = false)
 |-- geometry: geometry (nullable = true)
 |-- building_class: string (nullable = true)
 |-- height: double (nullable = true)
 |-- num_floors: integer (nullable = true)
 |-- burn_prob_mean: double (nullable = true)
 |-- burn_prob_max: double (nullable = true)
 |-- flame_length_mean: double (nullable = true)
 |-- wildfire_risk_class: string (nullable = true)
 |-- event_count_5km: long (nullable = true)
 |-- event_count_25km: long (nullable = true)
 |-- nearest_event_dist_m: double (nullable = true)
 |-- nearest_hail_m: double (nullable = true)
 |-- nearest_structure_m: double (nullable = true)
 |-- nearest_tvs_m: double (nullable = true)
 |-- hail_count_25km: long (nullable = true)
 |-- structure_count_25km: long (nullable = true)
 |-- tvs_count_25km: long (nullable = true)
 |-- max_hail_sevprob: double (nullable = true)
 |-- max_structure_reflectivity: double (nullable =

In [12]:
# ── 5b. Quick data quality check ──────────────────────────────────────────────
# Show coverage: what % of assets have non-null values for each hazard layer.
# Flood coverage is checked separately since it lives in its own weekly table.

total = asset_enriched.count()
wf_coverage = asset_enriched.filter(F.col("has_wildfire_data")).count()
sw_coverage = asset_enriched.filter(F.col("has_weather_data")).count()

FLOOD_SILVER = f"org_catalog.{SILVER_DB}.asset_flood_exposure"
fl_assets = sedona.sql(f"SELECT COUNT(DISTINCT asset_id) AS c FROM {FLOOD_SILVER}").collect()[0]["c"]

print(f"Total assets:                  {total:>10,}")
print(f"Wildfire exposure coverage:    {wf_coverage:>10,}  ({wf_coverage/total*100:.1f}%)")
print(f"Flood exposure coverage:       {fl_assets:>10,}  ({fl_assets/total*100:.1f}%)  (from weekly table)")
print(f"Severe weather coverage:       {sw_coverage:>10,}  ({sw_coverage/total*100:.1f}%)")
print()

# Distribution of wildfire risk classes
print("Wildfire risk class distribution:")
asset_enriched.groupBy("wildfire_risk_class").count().orderBy("count", ascending=False).show()

Total assets:                     357,263
Wildfire exposure coverage:       357,263  (100.0%)
Flood exposure coverage:          357,263  (100.0%)  (from weekly table)
Severe weather coverage:          357,263  (100.0%)

Wildfire risk class distribution:


,wildfire_risk_class,count
0,low,328416
1,moderate,20029
2,high,8583
3,very_high,235


In [13]:
# ── 5c. Write silver.asset_enriched to Iceberg ───────────────────────────────
ENRICHED_SILVER = f"org_catalog.{SILVER_DB}.asset_enriched"

asset_enriched.writeTo(ENRICHED_SILVER).createOrReplace()

row_count = sedona.table(ENRICHED_SILVER).count()
print(f"✓ Wrote {row_count:,} rows to {ENRICHED_SILVER}")

asset_enriched.unpersist()  # release cached DataFrame from memory


✓ Wrote 357,263 rows to org_catalog.silver.asset_enriched


,asset_id,asset_type,geometry,building_class,height,num_floors,burn_prob_mean,burn_prob_max,flame_length_mean,wildfire_risk_class,event_count_5km,event_count_25km,nearest_event_dist_m,nearest_hail_m,nearest_structure_m,nearest_tvs_m,hail_count_25km,structure_count_25km,tvs_count_25km,max_hail_sevprob,max_structure_reflectivity,max_tvs_delta_v,has_wildfire_data,has_weather_data,weather_window_start,weather_window_end,computed_at
0,423fd502-ee1b-403d-8d4b-cbf459086b75,building,,,nan,nan,9.999999747378752e-06,9.999999747378752e-06,0.0,low,11,12,633.397676359289,4318.899669668301,633.397676359289,nan,2,10,0,30.0,48.0,nan,True,True,2025-01-01,2026-03-25,2026-09-29 02:55:24.676204
1,4a54c158-86e6-4d44-a026-28b0b584d40b,building,,,nan,nan,0.0,0.0,0.0,low,11,16,393.6146930508434,2638.7571155662613,393.6146930508434,nan,6,10,0,30.0,46.0,nan,True,True,2025-01-01,2026-03-25,2026-09-29 02:55:24.676204
2,87aa02e2-ca66-4ac8-96cf-9a41cc170f16,building,,,3.038304090499878,nan,0.0,0.0,0.0,low,8,17,1380.961198592221,8626.470829960566,1380.961198592221,nan,7,10,0,30.0,47.0,nan,True,True,2025-01-01,2026-03-25,2026-09-29 02:55:24.676204
3,9fa796db-8f4d-4703-9c7e-79e70d72c1ae,building,,,3.0200719833374023,nan,0.0,0.0,0.0,low,8,17,1397.6994275823363,8617.178672513046,1397.6994275823363,nan,7,10,0,30.0,47.0,nan,True,True,2025-01-01,2026-03-25,2026-09-29 02:55:24.676204
4,c950cf68-2041-4fc2-af04-fe39534da6e3,building,,,2.67763352394104,nan,0.0,0.0,0.0,low,8,17,1431.1073002426358,8621.915505196013,1431.1073002426358,nan,7,10,0,30.0,47.0,nan,True,True,2025-01-01,2026-03-25,2026-09-29 02:55:24.676204
5,4ec5eb2b-5302-4cae-bcd9-6f1d6672b91c,building,,,nan,nan,0.0,0.0,0.0,low,11,12,644.5732375802669,4589.091345690617,644.5732375802669,nan,2,10,0,30.0,48.0,nan,True,True,2025-01-01,2026-03-25,2026-09-29 02:55:24.676204
6,9d4c4203-085e-4765-8cdc-01a300203fbd,building,,,2.8289785385131836,nan,0.0,0.0,0.0,low,8,17,1451.688808256515,8590.136374161471,1451.688808256515,nan,7,10,0,30.0,47.0,nan,True,True,2025-01-01,2026-03-25,2026-09-29 02:55:24.676204
7,1fa2976a-7c78-4e6f-be80-9a90715d52db,building,"<path fill-rule=""evenodd"" fill=""#66cc99"" stroke=""#555555"" stroke-width=""6.082559999924797e-06"" opacity=""0.6"" d=""M -117.0084753,32.9307301 L -117.008378,32.9308154 L -117.0084075,32.9308391 L -117.0083834,32.9308602 L -117.0084328,32.9309 L -117.0084516,32.9308835 L -117.0084792,32.9309058 L -117.0084835,32.930902 L -117.0085105,32.9309237 L -117.0085108,32.9309298 L -117.0085312,32.9309462 L -117.0085389,32.9309468 L -117.0085437,32.9309506 L -117.0085495,32.9309456 L -117.0085989,32.9309853 L -117.0086574,32.9309341 L -117.0086092,32.9308953 L -117.0086596,32.9308512 L -117.0085985,32.9308021 L -117.0085817,32.9308168 L -117.0085497,32.930791 L -117.008568,32.930775 L -117.0085285,32.9307432 L -117.0085108,32.9307587 L -117.0084753,32.9307301 z"" />",,5.204361438751221,nan,0.011264893226325512,0.012530163861811161,1.056455373764038,very_high,10,16,395.1044732655297,11992.032604074046,395.1044732655297,nan,6,10,0,-999.0,49.0,nan,True,True,2025-01-01,2026-03-25,2026-09-29 02:55:24.676204
8,4a9c57d1-b4ef-43f3-a776-83da8fe255f2,building,,,2.9915945529937744,nan,0.013522601686418056,0.013522601686418056,2.158792018890381,very_high,10,13,1083.833878394478,11322.61942411386,1083.833878394478,nan,3,10,0,-999.0,46.0,nan,True,True,2025-01-01,2026-03-25,2026-09-29 02:55:24.676204
9,5c6f16db-662b-4e53-8a8b-b5f4423a74c0,building,,,2.2834129333496094,nan,0.0,0.0,0.0,low,8,17,1332.9373539462745,8649.150965693321,1332.9373539462745,nan,7,10,0,30.0,47.0,nan,True,True,2025-01-01,2026-03-25,2026-09-29 02:55:24.676204


## 6. Verify Silver Layer

Final verification: list all Silver tables and confirm row counts.

In [14]:
# ── Verify all Silver tables ──────────────────────────────────────────────────
silver_tables = [
    "asset_wildfire_exposure",
    "asset_flood_exposure",
    "asset_weather_density",
    "asset_enriched",
]

print("Silver Layer Summary")
print("=" * 72)
for table in silver_tables:
    fqn = f"org_catalog.{SILVER_DB}.{table}"
    df = sedona.table(fqn)
    count = df.count()
    cols = len(df.columns)
    print(f"  {fqn:55s}  {count:>10,} rows  {cols:>3} cols")
print("=" * 72)
print("\n✓ Bronze → Silver pipeline complete.")
print(f"  Next step: run silver-to-gold.ipynb to produce industry-specific Gold tables.")

Silver Layer Summary
  org_catalog.silver.asset_wildfire_exposure                  357,263 rows   11 cols
  org_catalog.silver.asset_flood_exposure                   6,073,471 rows    8 cols
  org_catalog.silver.asset_weather_density                    357,263 rows   18 cols
  org_catalog.silver.asset_enriched                           357,263 rows   27 cols

✓ Bronze → Silver pipeline complete.
  Next step: run silver-to-gold.ipynb to produce industry-specific Gold tables.


26/09/29 03:40:59 ERROR RollingEventLogFilesWriter: Spark detects errors while closing event logs.
